# 04 - Feature Extraction
Stage 4 of the task pipeline (*Features / DL*). Input = the clean epochs saved by `main_pipeline.ipynb` in `outputs/`.

| Output | Used by |
|---|---|
| `outputs/cv_splits.csv` - one fixed hold-out test set + 5 CV folds, **split by recording session** | every model (same folds for all) |
| `outputs/features_bandpower.csv` - unsupervised spectral features | classical models, EDA of features |
| `bci_features.py` - FBCSP, band power, channel standardizer | modeling notebook + Streamlit app |

**Why split by session?** `main_pipeline` 7.3 showed the session explains much more of the mu power (eta² ≈ 0.3-0.4) than the label (≈ 0.001).
With a random split, trials from the same session land in train and test, and a model can score above chance by recognising the session.
The task asks for *subject/trial-aware* CV; the session is the closest grouping we have.

**Supervised features (CSP) are never fitted on all trials** - only inside each training fold.

## 1. Load the preprocessed epochs

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

import bci_features as bf
warnings.filterwarnings("ignore", category=FutureWarning)

OUT = "outputs"
with open(f"{OUT}/preprocessing_config.json") as fh:
    pcfg = json.load(fh)
SEED, FS, CH = pcfg["random_seed"], pcfg["fs"], pcfg["channels"]
assert FS == bf.FS and CH == bf.CHANNELS, "bci_features.py constants must match the preprocessing config"

X = np.load(f"{OUT}/epochs_CAR.npy")      # (trials, channels, samples), 0.5-40 Hz + notch + CAR, 1-9 s
y = np.load(f"{OUT}/labels.npy")          # 0 = LEFT, 1 = RIGHT
trials = pd.read_csv(f"{OUT}/trials_final.csv")
groups = trials["session"].values
assert len(X) == len(y) == len(trials)

print("X:", X.shape, "| y:", np.bincount(y).tolist(), "[LEFT, RIGHT] | sessions:", len(np.unique(groups)))
print(f"Epoch length: {X.shape[-1] / FS:.1f} s @ {FS} Hz | channels: {CH}")

## 2. Fixed splits (shared by all models)
1. **Hold-out test** ≈ 20% of the sessions - touched only once, at the very end.
2. **5-fold CV** on the remaining sessions - for model selection and tuning.

No session appears in two places.

In [ ]:
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
dev_idx, test_idx = next(outer.split(X, y, groups))

splits = trials[["trial_id", "filename", "label", "session"]].copy()
splits["split"] = "dev"
splits.loc[test_idx, "split"] = "test"
splits["fold"] = -1

inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
for k, (_, va) in enumerate(inner.split(X[dev_idx], y[dev_idx], groups[dev_idx])):
    splits.loc[dev_idx[va], "fold"] = k

assert not set(groups[dev_idx]) & set(groups[test_idx])
splits.to_csv(f"{OUT}/cv_splits.csv", index=False)

summary = (splits.assign(part=np.where(splits["split"] == "test", "test", "fold " + splits["fold"].astype(str)))
           .groupby("part").agg(trials=("label", "size"), sessions=("session", "nunique"),
                                RIGHT_share=("label", lambda s: (s == "RIGHT").mean().round(3))))
display(summary)
print("Saved outputs/cv_splits.csv")

## 3. Band-power features (unsupervised)
Per trial, from the Welch PSD (2 s windows):
- **log band power** in mu (8-12 Hz) and beta (13-30 Hz), per channel
- **relative band power** = band / total 4-40 Hz power (removes the overall amplitude, which differs a lot between sessions)
- **asymmetry** log(C3) - log(C4) per band - motor imagery should push it in opposite directions for LEFT and RIGHT

No labels are used, so computing them on all trials is not leakage.

In [ ]:
bp = bf.BandPower()
F_bp = pd.DataFrame(bp.transform(X), columns=bp.get_feature_names_out())
F_bp.insert(0, "trial_id", trials["trial_id"].values)
F_bp.to_csv(f"{OUT}/features_bandpower.csv", index=False)
print("Band-power features:", F_bp.shape[1] - 1, "-> saved outputs/features_bandpower.csv")
display(F_bp.describe().T.round(3).head(10))

### 3.1 How much does each feature know about the label vs the session?
- **AUC** of a single feature for LEFT vs RIGHT (0.5 = no information; computed on the dev set only)
- **eta² label / eta² session** = share of the feature's variance explained by the label vs by the session

In [ ]:
def eta2(v, g):
    s = pd.Series(v).groupby(g)
    return float(((s.mean() - v.mean()) ** 2 * s.size()).sum() / ((v - v.mean()) ** 2).sum())

feat_cols = F_bp.columns[1:]
Fd, yd, gd = F_bp.loc[dev_idx, feat_cols], y[dev_idx], groups[dev_idx]
uni = pd.DataFrame({
    "AUC": [roc_auc_score(yd, Fd[c]) for c in feat_cols],
    "t-test p": [stats.ttest_ind(Fd[c][yd == 0], Fd[c][yd == 1]).pvalue for c in feat_cols],
    "eta² label": [eta2(Fd[c].values, yd) for c in feat_cols],
    "eta² session": [eta2(Fd[c].values, gd) for c in feat_cols],
}, index=feat_cols)
uni["|AUC-0.5|"] = (uni["AUC"] - 0.5).abs()
display(uni.sort_values("|AUC-0.5|", ascending=False).round(4))

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
uni["AUC"].plot.bar(ax=ax[0]); ax[0].axhline(0.5, color="k", lw=1)
ax[0].set_title("Single-feature AUC (LEFT vs RIGHT)"); ax[0].set_ylim(0.4, 0.6)
uni[["eta² label", "eta² session"]].plot.bar(ax=ax[1], logy=True)
ax[1].set_title("Variance explained: label vs session")
plt.tight_layout(); plt.show()

print(f"Best single-feature AUC: {uni['AUC'].iloc[uni['|AUC-0.5|'].argmax()]:.3f} ({uni['|AUC-0.5|'].idxmax()})")
print(f"Median eta² session / eta² label: {np.median(uni['eta² session'] / uni['eta² label'].clip(1e-9)):.0f}x")

### 3.2 Asymmetry LEFT vs RIGHT
The clearest physiological marker: imagining the RIGHT hand should lower mu power over C3 (left hemisphere), so log(C3) - log(C4) should be **lower for RIGHT**.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, b in zip(ax, bf.BANDS):
    col = f"Asym_{b}_C3-C4"
    for lab, name, c in [(0, "LEFT", "tab:blue"), (1, "RIGHT", "tab:red")]:
        a.hist(Fd[col][yd == lab], bins=50, alpha=0.5, density=True, color=c, label=name)
    d = (Fd[col][yd == 1].mean() - Fd[col][yd == 0].mean()) / Fd[col].std()
    a.set_title(f"{col}  (RIGHT-LEFT d = {d:+.3f}, expected < 0)"); a.legend()
plt.tight_layout(); plt.show()

## 4. Filter-Bank CSP (supervised -> fitted per fold)
CSP finds spatial filters whose output variance is high for one class and low for the other. FBCSP repeats it in 5 sub-bands
(8-12, 12-16, 16-20, 20-24, 24-30 Hz). With 4 channels we keep 1 pair per band -> 10 features.

Below, CSP is fitted on the training part of each fold and the features are computed on the held-out fold
(**out-of-fold**), so the AUCs are honest.

In [ ]:
fbcsp_names = bf.FilterBankCSP().get_feature_names_out()
F_csp_oof = np.full((len(dev_idx), len(fbcsp_names)), np.nan)
fold_dev = splits.loc[dev_idx, "fold"].values
patterns = []
for k in range(5):
    tr, va = fold_dev != k, fold_dev == k
    fb = bf.FilterBankCSP().fit(X[dev_idx][tr], yd[tr])
    F_csp_oof[va] = fb.transform(X[dev_idx][va])
    patterns.append(fb.csps_[0].patterns_)   # mu-band patterns for plotting

csp_auc = pd.DataFrame({
    "OOF AUC": [roc_auc_score(yd, F_csp_oof[:, j]) for j in range(len(fbcsp_names))],
    "train-fold AUC (fold 0)": None,
}, index=fbcsp_names)
fb0 = bf.FilterBankCSP().fit(X[dev_idx][fold_dev != 0], yd[fold_dev != 0])
F0 = fb0.transform(X[dev_idx][fold_dev != 0])
csp_auc["train-fold AUC (fold 0)"] = [roc_auc_score(yd[fold_dev != 0], F0[:, j]) for j in range(F0.shape[1])]
display(csp_auc.round(3))
print("A big gap between train-fold and out-of-fold AUC = CSP is fitting noise / session, not the task.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
P = np.array(patterns)                         # (folds, 2 filters, channels)
for j, a in enumerate(ax):
    for k in range(len(P)):
        v = P[k, j] / np.abs(P[k, j]).max()
        a.plot(CH, v * np.sign(v[np.abs(v).argmax()]), marker="o", alpha=0.7, label=f"fold {k}")
    a.axhline(0, color="k", lw=0.8)
    a.set_title(f"Mu-band CSP pattern {j + 1} (normalised)")
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
print("Stable patterns across folds focused on C3 / C4 = good sign. Patterns that jump around = no consistent spatial effect.")

## 5. Input for the deep models
EEGNet / CNN take the epochs directly: `(trials, 1, channels, samples)`.
`bf.ChannelStandardizer` is fitted on the training trials only and stored with the model, so the Streamlit app scales new trials the same way.

In [ ]:
tr_mask = fold_dev != 0
scaler = bf.ChannelStandardizer().fit(X[dev_idx][tr_mask])
X_dl = scaler.transform(X[dev_idx][~tr_mask])[:, None]          # add the "image" axis for Conv2d
print("DL tensor example (fold 0 validation):", X_dl.shape, X_dl.dtype,
      "| per-channel mean ≈ 0:", np.round(X_dl.mean(axis=(0, 1, 3)), 2))

## 6. Save the feature config

In [ ]:
feat_cfg = {
    "source": "outputs/epochs_CAR.npy (main_pipeline.ipynb)",
    "fs": FS, "channels": CH, "epoch_seconds": pcfg["epoch_seconds"],
    "bandpower_bands": bf.BANDS, "fbcsp_bands": bf.FB_BANDS, "csp_pairs_per_band": 1,
    "n_features": {"bandpower": len(feat_cols), "fbcsp": len(fbcsp_names)},
    "splits": {"method": "StratifiedGroupKFold by session", "seed": SEED,
               "dev_trials": int(len(dev_idx)), "test_trials": int(len(test_idx)), "cv_folds": 5},
}
with open(f"{OUT}/features_config.json", "w") as fh:
    json.dump(feat_cfg, fh, indent=2)
print(json.dumps(feat_cfg, indent=2))

## Summary
- **Splits:** fixed and session-grouped, in `outputs/cv_splits.csv`. Every model in `05_Modeling` must use them.
- **Features:** 18 band-power features (no labels) + 10 FBCSP features (fitted per fold), all in `bci_features.py`.
- **What to read before modeling:** single-feature AUCs (3.1) and out-of-fold CSP AUCs (4). If they all stay around 0.5 and
  eta² session ≫ eta² label, expect near-chance accuracy - consistent with the EDA decision in `main_pipeline` 6.5 and the label-shift finding in 7.6.

**Next:** `05_Modeling.ipynb` - CSP+SVM baseline -> EEGNet -> CNN, all on the same splits, compared with a permutation test.